Import Libraries& Connect to NEO4J

In [32]:
import pandas as pd
from neo4j import GraphDatabase

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [2]:
uri = "neo4j://127.0.0.1:7687"
username = "neo4j"
password = "juilee1609"

driver = GraphDatabase.driver(uri, auth=(username, password))

def run_query(query):
    with driver.session() as session:
        return [r.data() for r in session.run(query)]

In [6]:
df = pd.read_csv('C:/Users/Juilee/Desktop/Big data and Bussiness Intelligence Capstone Project/data/merged_df.csv')

In [33]:
query = """
MATCH (o:Order)
RETURN 
    o.id AS node_id,
    COUNT { (o)--() } AS degree
"""

degree_df = pd.DataFrame(run_query(query))
degree_df.head()

,node_id,degree
0,77202,3
1,75939,3
2,75938,3
3,75937,3
4,75936,3


In [11]:
df.rename(columns={"order_id": "node_id"}, inplace=True)
degree_df['node_id'] = degree_df['node_id'].astype('int64')
df_final = df.merge(degree_df, on="node_id", how="left")
df_final.head()
#s5_matrix = df.merge(degree_df, on="node_id", how="left")

,node_id,customer_id,product_name,category,department,Product Price,product_image,Order Region,Market,Order Status,...,quantity,profit,Latitude,Longitude,delay,view_count,unique_users,peak_month,peak_hour,degree
0,77202,20755,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,Southeast Asia,Pacific Asia,COMPLETE,...,1,91.250000,18.251453,-66.037056,-1.0,0.0,0.0,NaN,NaN,3
1,75939,19492,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,South Asia,Pacific Asia,PENDING,...,1,-249.089996,18.279451,-66.037064,1.0,0.0,0.0,NaN,NaN,3
2,75938,19491,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,South Asia,Pacific Asia,CLOSED,...,1,-247.779999,37.292233,-121.881279,0.0,0.0,0.0,NaN,NaN,3
3,75937,19490,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,Oceania,Pacific Asia,COMPLETE,...,1,22.860001,34.125946,-118.291016,-1.0,0.0,0.0,NaN,NaN,3
4,75936,19489,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,Oceania,Pacific Asia,PENDING_PAYMENT,...,1,134.210007,18.253769,-66.037048,-2.0,0.0,0.0,NaN,NaN,3


In [16]:
X_base = df_final[[
    "sales",
    "Shipping Mode",
    "Order Region",
    "category",
    "degree"
]]

y = df_final["delay"]

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X_base,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [18]:
num_features = ["sales", "degree"]
cat_features = ["Shipping Mode", "Order Region", "category"]

baseline_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

In [19]:
baseline_model = Pipeline([
    ("preprocessor", baseline_preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [20]:
baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

baseline_report = classification_report(
    y_test,
    baseline_pred,
    output_dict=True
)

print(classification_report(y_test, baseline_pred))

              precision    recall  f1-score   support

        -2.0       0.21      0.24      0.22      4333
        -1.0       0.21      0.20      0.20      4340
         0.0       0.26      0.26      0.26      6751
         1.0       0.61      0.58      0.59     12129
         2.0       0.21      0.22      0.21      5744
         3.0       0.21      0.21      0.21      1410
         4.0       0.21      0.20      0.20      1397

    accuracy                           0.35     36104
   macro avg       0.27      0.27      0.27     36104
weighted avg       0.35      0.35      0.35     36104



In [21]:
query = """
MATCH (o:Order)
RETURN
    o.id AS node_id,
    o.pagerank AS pagerank,
    o.community AS community
"""

gds_features = pd.DataFrame(run_query(query))

In [23]:
df_final["node_id"] = df_final["node_id"].astype(str)

gds_features["node_id"] = gds_features["node_id"].astype(str)

In [24]:
enriched_matrix = pd.merge(
    df_final,
    gds_features,
    how="left",
    on="node_id"
)

In [25]:
enriched_matrix["pagerank"] = enriched_matrix["pagerank"].fillna(0)

enriched_matrix["community"] = enriched_matrix["community"].fillna(-1)

enriched_matrix["community"] = enriched_matrix["community"].astype(int)

In [26]:
X_enriched = enriched_matrix[[
    "sales",
    "Shipping Mode",
    "Order Region",
    "category",
    "degree",
    "pagerank",
    "community"
]]

In [27]:
num_features_enriched = ["sales", "degree", "pagerank"]

cat_features_enriched = [
    "Shipping Mode",
    "Order Region",
    "category",
    "community"
]

enriched_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features_enriched),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features_enriched)
])

In [28]:
enriched_model = Pipeline([
    ("preprocessor", enriched_preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [29]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_enriched,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

enriched_model.fit(X_train2, y_train2)

enriched_pred = enriched_model.predict(X_test2)

enriched_report = classification_report(
    y_test2,
    enriched_pred,
    output_dict=True
)

print(classification_report(y_test2, enriched_pred))

              precision    recall  f1-score   support

        -2.0       0.26      0.26      0.26      4333
        -1.0       0.27      0.27      0.27      4340
         0.0       0.35      0.35      0.35      6751
         1.0       0.64      0.63      0.64     12129
         2.0       0.27      0.29      0.28      5744
         3.0       0.31      0.31      0.31      1410
         4.0       0.31      0.30      0.30      1397

    accuracy                           0.41     36104
   macro avg       0.34      0.34      0.34     36104
weighted avg       0.41      0.41      0.41     36104



In [30]:
comparison = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1-score"],
    
    "Baseline": [
        baseline_report["weighted avg"]["precision"],
        baseline_report["weighted avg"]["recall"],
        baseline_report["weighted avg"]["f1-score"]
    ],
    
    "Enriched": [
        enriched_report["weighted avg"]["precision"],
        enriched_report["weighted avg"]["recall"],
        enriched_report["weighted avg"]["f1-score"]
    ]
})

comparison["Delta"] = (
    comparison["Enriched"] - comparison["Baseline"]
)

comparison

,Metric,Baseline,Enriched,Delta
0,Precision,0.353935,0.411318,0.057383
1,Recall,0.346333,0.409318,0.062985
2,F1-score,0.349808,0.410268,0.060460
